In [ ]:
# %% [markdown]
# # Task 4: Churn Prediction Model
# 
# **Using data prepared in Excel (Tasks 1-3)**

# %% [code]
# ============================================
# 1. IMPORT LIBRARIES
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                            f1_score, confusion_matrix, classification_report,
                            roc_auc_score, roc_curve)
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Libraries imported successfully!")

# %% [markdown]
# ## 2. LOAD YOUR DATA

# %% [code]
# Load your prepared dataset - clean CSV, no extra rows!
df = pd.read_csv('data/Telco_Churn_Task1_Prepared.csv')

print("="*60)
print("DATASET INFORMATION")
print("="*60)
print(f"Dataset shape: {df.shape}")
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

print("\nFirst 5 rows:")
print(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes.value_counts())

# %% [markdown]
# ## 3. CHECK DATA READY STATUS

# %% [code]
print("\n" + "="*60)
print("CHECKING DATA READY STATUS")
print("="*60)

# Check if Churn column exists and is numeric
if 'Churn' in df.columns:
    print(f"✅ Found Churn column!")
    print(f"Churn column type: {df['Churn'].dtype}")
    print(f"Churn unique values: {df['Churn'].unique()}")
    print(f"Churn rate: {df['Churn'].mean():.2%}")
else:
    print("❌ Churn column not found!")
    raise KeyError("Churn column not found")

# Check for any text columns
text_cols = df.select_dtypes(include=['object']).columns.tolist()
if text_cols:
    print(f"\n⚠️ These columns still have text: {text_cols}")
    print("Converting to numeric...")
    for col in text_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    print("✅ All text columns converted to numeric")
else:
    print("\n✅ All columns are numeric - ready for modeling!")

# %% [markdown]
# ## 4. SEPARATE FEATURES AND TARGET

# %% [code]
# Drop columns we don't need
columns_to_drop = ['Tenure Group', 'Segment']
for col in columns_to_drop:
    if col in df.columns:
        df = df.drop(col, axis=1)
        print(f"✅ Dropped {col}")

# Separate features and target
X = df.drop('Churn', axis=1)
y = df['Churn']

print("\n" + "="*60)
print("FEATURES AND TARGET")
print("="*60)
print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

print(f"\nTarget distribution:")
print(y.value_counts())
print(f"\nChurn rate: {y.mean():.2%}")

# %% [markdown]
# ## 5. TRAIN-TEST SPLIT

# %% [code]
# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("="*60)
print("TRAIN-TEST SPLIT")
print("="*60)
print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")
print(f"\nTraining set churn rate: {y_train.mean():.2%}")
print(f"Testing set churn rate: {y_test.mean():.2%}")

# %% [markdown]
# ## 6. FEATURE SCALING

# %% [code]
# Identify numerical columns that need scaling
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# Only scale if these columns exist
numerical_cols = [col for col in numerical_cols if col in X.columns]

# Create scaler
scaler = StandardScaler()

# Scale numerical features
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

if numerical_cols:
    X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
    X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])
    print("="*60)
    print("FEATURE SCALING COMPLETE")
    print("="*60)
    print(f"Scaled features: {numerical_cols}")
    print("✅ Scaler ready")
else:
    print("⚠️ No numerical columns to scale")

# %% [markdown]
# ## 7. MODEL EVALUATION FUNCTION

# %% [code]
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """
    Train and evaluate a model with comprehensive metrics
    """
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    # Cross-validation score
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1')
    
    # Print results
    print(f"\n{'='*50}")
    print(f"{model_name} Results:")
    print(f"{'='*50}")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")
    print(f"CV F1-Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    print(f"\nConfusion Matrix:")
    print(f"TN: {cm[0,0]:4d}  FP: {cm[0,1]:4d}")
    print(f"FN: {cm[1,0]:4d}  TP: {cm[1,1]:4d}")
    
    return {
        'model': model,
        'model_name': model_name,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba,
        'confusion_matrix': cm,
        'y_test': y_test
    }

# %% [markdown]
# ## 8. TRAIN MODELS

# %% [code]
# Logistic Regression
print("\n" + "="*60)
print("TRAINING LOGISTIC REGRESSION")
print("="*60)

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'
)

lr_results = evaluate_model(
    lr_model, X_train_scaled, X_test_scaled, 
    y_train, y_test, 
    "Logistic Regression"
)

# %% [code]
# Decision Tree
print("\n" + "="*60)
print("TRAINING DECISION TREE")
print("="*60)

dt_model = DecisionTreeClassifier(
    random_state=42,
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight='balanced'
)

dt_results = evaluate_model(
    dt_model, X_train, X_test, 
    y_train, y_test, 
    "Decision Tree"
)

# %% [code]
# Random Forest
print("\n" + "="*60)
print("TRAINING RANDOM FOREST")
print("="*60)

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=15,
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight='balanced',
    n_jobs=-1
)

rf_results = evaluate_model(
    rf_model, X_train, X_test, 
    y_train, y_test, 
    "Random Forest"
)

# %% [code]
# Gradient Boosting
print("\n" + "="*60)
print("TRAINING GRADIENT BOOSTING")
print("="*60)

gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42,
    subsample=0.8
)

gb_results = evaluate_model(
    gb_model, X_train, X_test, 
    y_train, y_test, 
    "Gradient Boosting"
)

# %% [markdown]
# ## 9. MODEL COMPARISON

# %% [code]
# Create comparison DataFrame
comparison_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest', 'Gradient Boosting'],
    'Accuracy': [lr_results['accuracy'], dt_results['accuracy'], 
                 rf_results['accuracy'], gb_results['accuracy']],
    'Precision': [lr_results['precision'], dt_results['precision'], 
                  rf_results['precision'], gb_results['precision']],
    'Recall': [lr_results['recall'], dt_results['recall'], 
               rf_results['recall'], gb_results['recall']],
    'F1-Score': [lr_results['f1'], dt_results['f1'], 
                 rf_results['f1'], gb_results['f1']],
    'ROC-AUC': [lr_results['roc_auc'], dt_results['roc_auc'], 
                rf_results['roc_auc'], gb_results['roc_auc']]
})

print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)
print(comparison_df.to_string(index=False))

# Best model
best_model_row = comparison_df.loc[comparison_df['F1-Score'].idxmax()]
print(f"\n{'='*50}")
print(f"🏆 BEST MODEL: {best_model_row['Model']}")
print(f"{'='*50}")
print(f"F1-Score:  {best_model_row['F1-Score']:.4f}")
print(f"Accuracy:  {best_model_row['Accuracy']:.4f}")
print(f"Precision: {best_model_row['Precision']:.4f}")
print(f"Recall:    {best_model_row['Recall']:.4f}")
print(f"ROC-AUC:   {best_model_row['ROC-AUC']:.4f}")

# %% [markdown]
# ## 10. FEATURE IMPORTANCE

# %% [code]
print("\n" + "="*60)
print("FEATURE IMPORTANCE")
print("="*60)

feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10).to_string(index=False))

# Create directories
os.makedirs('reports/figures', exist_ok=True)

# Visualize
plt.figure(figsize=(12, 8))
top_10 = feature_importance.head(10)
plt.barh(top_10['feature'], top_10['importance'])
plt.xlabel('Feature Importance')
plt.title('Top 10 Most Important Features for Churn Prediction')
plt.tight_layout()
plt.savefig('reports/figures/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

# %% [markdown]
# ## 11. SAVE MODEL

# %% [code]
os.makedirs('models', exist_ok=True)

# Save the best model (Random Forest)
joblib.dump(rf_model, 'models/best_churn_model.pkl')
joblib.dump(scaler, 'models/scaler.pkl')

print("✅ Models saved successfully!")
print(f"- Best model: models/best_churn_model.pkl")
print(f"- Scaler: models/scaler.pkl")

# Save comparison
comparison_df.to_csv('reports/model_comparison_results.csv', index=False)
feature_importance.to_csv('reports/feature_importance.csv', index=False)

print(f"\n✅ Results saved to reports/")

# %% [markdown]
# ## 12. BUSINESS RECOMMENDATIONS

# %% [code]
print("\n" + "="*60)
print("💡 BUSINESS RECOMMENDATIONS")
print("="*60)

print("\nTop 5 Most Important Factors for Churn:")
for idx, row in feature_importance.head(5).iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")

print("\nKey Recommendations:")
print("-" * 50)
print("1. Convert month-to-month customers to yearly contracts")
print("2. Focus retention efforts on customers with < 12 months tenure")
print("3. Encourage automatic payment methods (discounts)")
print("4. Offer tech support and security as value-adds")
print("5. Create targeted retention campaigns for high-risk segments")

print("\n" + "="*60)
print("🎉 TASK 4 COMPLETE!")
print("="*60)

✅ Libraries imported successfully!
DATASET INFORMATION
Dataset shape: (7042, 43)
Number of rows: 7042
Number of columns: 43

First 5 rows:
   0  0.1  1  0.2  1.1  0.3  1.2  29.85  29.85.1 1.3  ...  1.11  0.20  0.21  \
0  1    0  0    0   34    1    0  56.95  1889.50   0  ...     0     1     0   
1  1    0  0    0    2    1    1  53.85   108.15   0  ...     1     0     0   
2  1    0  0    0   45    0    0  42.30  1840.75   1  ...     0     1     0   
3  0    0  0    0    2    1    1  70.70   151.65  x0  ...     1     0     0   
4  0    0  0    0    8    1    1  99.65   820.50   0  ...     1     0     0   

   1.12  0.22  0.23  0.24  0.25   0-12 Months  Unnamed: 42  
0     0     1     0     0     0  61-72 Months          NaN  
1     0     1     0     0     1   0-12 Months          NaN  
2     0     0     1     0     0  25-36 Months          NaN  
3     1     0     0     0     1   0-12 Months          NaN  
4     1     0     0     0     1  37-48 Months          NaN  

[5 rows x 43 column

KeyError: 'Churn column not found'